<a href="https://colab.research.google.com/github/suvigyajain0101/Large-Audio-Model-LAM/blob/main/notebooks/JAX_Foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01: JAX Foundations for Large Audio Models (LAMs)

Here we build a foundation in **research-grade JAX**.

---
## Module 1 (Refresher): Functional Foundations

To scale seamlessly across GPUs and TPUs via the **XLA (Accelerated Linear Algebra)** compiler, JAX enforces three functional programming rules:

1. **Immutable Arrays (`.at[].set()`)**:
   In NumPy/PyTorch, `x[0] = 5` mutates memory in place. In JAX, arrays are immutable. You express updates functionally via `y = x.at[idx].set(val)` (or `.add()`, `.multiply()`). Inside `@jax.jit`, XLA's memory aliasing pass automatically compiles this into an in-place hardware buffer update whenever `x` is no longer needed.
2. **Pure Functions**:
   Every JIT-compiled or transformed function must depend *only* on its explicit arguments and produce *zero side effects* (no mutating globals).
3. **Explicit PRNG State (`jax.random.split`)**:
   Instead of a hidden global random seed, JAX passes an explicit `key = jax.random.PRNGKey(seed)`. Every time you need randomness, you **split** the key (`key, subkey = jax.random.split(key)`) and pass `subkey` as the **first argument** to the random op.

In [3]:
import jax
import jax.numpy as jnp

# 1.1 Vectorized One-Hot Encoding
# In Audio Tokenizers (VQ / BEST-RQ), we map discrete codebook IDs (e.g. [3, 0, 2, 1])
# into one-hot vectors using functional .at[row_indices, col_indices].set(1.0)
def custom_one_hot(labels: jax.Array, num_classes: int) -> jax.Array:
    """Vectorized one-hot encoding using functional .at[row, col].set(1.0)."""
    init_array = jnp.zeros(shape=(len(labels), num_classes), dtype=jnp.float32)
    row_indices = jnp.arange(len(labels))
    return init_array.at[row_indices, labels].set(1.0)


# 1.2 Stateless Inverted Dropout with Explicit PRNG Key Splitting
def dropout_layer(x: jax.Array, key: jax.Array, drop_rate: float = 0.5) -> tuple[jax.Array, jax.Array]:
    """Applies inverted dropout and returns (scaled_output, next_key)."""
    mask_key, next_key = jax.random.split(key, 2)
    keep_prob = 1.0 - drop_rate
    keep_mask = jax.random.bernoulli(mask_key, p=keep_prob, shape=x.shape).astype(x.dtype)
    scaled_x = (x * keep_mask) / keep_prob
    return scaled_x, next_key


# Quick Verification
audio_codebook_ids = jnp.array([3, 0, 2, 1])
one_hot_codes = custom_one_hot(audio_codebook_ids, num_classes=4)
print("One-hot encoded audio tokens:\n", one_hot_codes)

rng_key = jax.random.PRNGKey(42)
sample_features = jnp.ones((4, 8))
dropped_features, rng_key = dropout_layer(sample_features, rng_key, drop_rate=0.25)
print("\nDropped features:\n", dropped_features)
print("Updated PRNG key:", rng_key)

One-hot encoded audio tokens:
 [[0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]]

Dropped features:
 [[1.3333334 1.3333334 0.        1.3333334 1.3333334 1.3333334 1.3333334
  1.3333334]
 [1.3333334 1.3333334 1.3333334 1.3333334 0.        1.3333334 1.3333334
  1.3333334]
 [1.3333334 1.3333334 1.3333334 0.        1.3333334 1.3333334 1.3333334
  0.       ]
 [1.3333334 0.        0.        0.        0.        1.3333334 1.3333334
  1.3333334]]
Updated PRNG key: [  64467757 2916123636]


---
## Module 2 (Refresher): Core Transformations (`@jit`, `vmap`, `value_and_grad`)

1. **`@jax.jit(static_argnames=[...])`**:
   Traces your function using abstract `Tracer` objects (which carry `shape` and `dtype`, but *not* concrete runtime values) and compiles it into fused XLA machine code. Any argument used in a Python `if` condition or to determine array shapes must be listed in `static_argnames`.
2. **`jax.vmap(fn, in_axes=...)`**:
   Write your function for a **single example** (e.g., a single audio frame or a single query vector), and let `jax.vmap` vectorize it across a batch automatically. Use `0` for batched inputs and `None` for shared parameters/matrices.
3. **`jax.value_and_grad(loss_fn, argnums=0)`**:
   Computes both the scalar loss and parameter gradients $\nabla_w \mathcal{L}(w)$ in a single fused forward-backward pass.

In [5]:
# 2.1 Automatic Vectorization (vmap): Single-Frame Query Attention -> Multi-Frame Attention
# Suppose our Conformer outputs 25 audio frames per second (1 frame every 40ms), each of dimension d_model=64.
def compute_query_score(q_single: jax.Array, K: jax.Array) -> jax.Array:
    """Computes scaled dot-product attention scores for ONE query (d_model,) against K (seq_len, d_model)."""
    d_model = q_single.shape[-1]
    return jnp.dot(K, q_single) / jnp.sqrt(d_model)

# Vectorize over axis 0 of Q (num_queries, d_model), keeping K (seq_len, d_model) unbatched (None)
batched_compute_scores = jax.jit(jax.vmap(compute_query_score, in_axes=(0, None)))


# 2.2 JIT with Static Arguments & Single-Pass Value + Gradient
@jax.jit(static_argnames=["use_l2"])
def compute_regularized_loss(
    weights: jax.Array, X: jax.Array, y: jax.Array, use_l2: bool, reg_lambda: float = 0.01
) -> jax.Array:
    preds = jnp.dot(X, weights)
    mse_loss = jnp.mean((preds - y) ** 2)
    # Because `use_l2` is in static_argnames, Python can branch on it at trace time!
    if use_l2:
        l2_penalty = reg_lambda * jnp.sum(weights ** 2)
        return mse_loss + l2_penalty
    return mse_loss

# Differentiate with respect to argument 0 (`weights`)
loss_and_grad_fn = jax.value_and_grad(compute_regularized_loss, argnums=0)


# Quick Verification
Q_audio_frames = jnp.ones((25, 64))   # 25 audio frames (1 second at 40ms/frame), d_model=64
K_audio_frames = jnp.ones((25, 64))
attn_logits = batched_compute_scores(Q_audio_frames, K_audio_frames)
print("Attention logits shape (25 queries x 25 keys):", attn_logits.shape)

w_init = jnp.zeros((64,))
dummy_y = jnp.ones((25,))
loss_val, w_grads = loss_and_grad_fn(w_init, Q_audio_frames, dummy_y, True)
print(f"Initial Loss: {loss_val:.4f} | Grads shape: {w_grads.shape} | Grads norm: {jnp.linalg.norm(w_grads):.4f}")

Attention logits shape (25 queries x 25 keys): (25, 25)
Initial Loss: 1.0000 | Grads shape: (64,) | Grads norm: 16.0000


---
## Module 3 (New!): Structured Control Flow (`jax.lax.scan` & `jax.lax.cond`)

### Why Python `for` Loops Are Fatal for Sequential Audio & Agentic Loops
Suppose we have a 30-second audio clip represented at `25 Hz` (`750` frames) or `16,000 Hz` (`480,000` raw samples), and we want to run a stateful operation across time $t = 1 \dots T$ (such as a digital audio filter, a streaming KV-cache update, or a recurrent state).

If you write a standard Python `for t in range(T):` loop inside `@jax.jit`:
* **JAX unrolls the entire loop during tracing!**
* If $T = 750$, XLA creates 750 duplicate copies of your layer in the computation graph. Compilation takes minutes instead of milliseconds, and TPU/GPU instruction cache overflows.

### The Solution: `jax.lax.scan(step_fn, init_carry, xs)`
`jax.lax.scan` is JAX's compiled sequential loop primitive. It compiles `step_fn` **only once** and executes it in a tight hardware loop at C++/XLA speed:

```python
def step_fn(carry, x_t):
    # carry: state passed from step t-1
    # x_t:   slice of input sequence at step t
    new_carry = ...
    y_t = ...
    return new_carry, y_t

final_carry, ys_stacked = jax.lax.scan(step_fn, init_carry, xs)

In [6]:
@jax.jit
def compute_audio_envelope(waveform: jax.Array, alpha: float = 0.95) -> tuple[jax.Array, jax.Array]:
    """Tracks the smooth amplitude envelope of a 1D waveform using jax.lax.scan."""

    # step_fn always takes (carry, x_t) and returns (new_carry, y_t)
    def ema_step(prev_env: jax.Array, sample_t: jax.Array) -> tuple[jax.Array, jax.Array]:
        current_env = alpha * prev_env + (1.0 - alpha) * jnp.abs(sample_t)
        # First return value is passed as `prev_env` to step t+1;
        # Second return value is collected into the output array `envelope_curve`!
        return current_env, current_env

    init_env = jnp.array(0.0, dtype=waveform.dtype)
    final_env, envelope_curve = jax.lax.scan(ema_step, init_env, waveform)
    return final_env, envelope_curve


# Test on a synthetic 1-second 16kHz burst of sound (440 Hz tone active between t=0.2s and t=0.6s)
t = jnp.linspace(0.0, 1.0, 16000)
burst_waveform = jnp.sin(2 * jnp.pi * 440.0 * t) * (t > 0.2) * (t < 0.6)

final_state, env = compute_audio_envelope(burst_waveform, alpha=0.99)
print("Waveform shape:", burst_waveform.shape, "| Envelope shape:", env.shape)
print(f"Peak envelope during 440Hz tone: {jnp.max(env):.4f} | Final envelope at t=1.0s: {final_state:.6f}")

Waveform shape: (16000,) | Envelope shape: (16000,)
Peak envelope during 440Hz tone: 0.6490 | Final envelope at t=1.0s: 0.000000


---
## Module 3 Practice Problems

Solve the following two audio-inspired problems using `jax.lax.scan` and `jax.lax.cond`.

### Problem 3.1: Pre-Emphasis Audio Filter with `jax.lax.scan`
Before computing a Mel-Spectrogram for speech recognition or USM tokenization, audio pipelines often apply a **first-order pre-emphasis filter** to boost high-frequency consonant energy ($4\text{–}8\text{ kHz}$, like *"s"*, *"t"*, *"f"*) relative to loud low-frequency vowels:
$$y_t = x_t - \alpha \cdot x_{t-1} \quad (\text{typically } \alpha = 0.97, \text{ with } x_{-1} = 0.0)$$

**Task**: Implement `pre_emphasis_scan(waveform, alpha=0.97)` using `jax.lax.scan`.
* *Hint*: What needs to be passed from step $t-1$ to step $t$ as the `carry`? Just the previous raw sample $x_{t-1}$!
* At each step $t$, given `carry = x_prev` and input `x_t`, compute `y_t = x_t - alpha * x_prev`, and return `(x_t, y_t)` so that `x_t` becomes the `carry` (`x_prev`) for step $t+1$!

---

### Problem 3.2: Dynamic Audio Noise Gate (VAD) with `jax.lax.cond` + `jax.vmap`
Suppose we have a sequence of audio frames `frames` of shape `(num_frames, frame_len)` (for example, `25` frames of `640` samples each for 1 second of `16 kHz` audio).
For a **single frame** `frame` of shape `(frame_len,)`:
1. Compute its Root-Mean-Square (RMS) energy: $\text{rms} = \sqrt{\frac{1}{L} \sum_{i=1}^L \text{frame}_i^2}$ (`jnp.sqrt(jnp.mean(frame ** 2))`).
2. Using `jax.lax.cond`, if `rms >= threshold`, return the `frame` unchanged; if `rms < threshold` (silence/background hiss), return an array of zeros `jnp.zeros_like(frame)`.
3. Finally, use `jax.vmap` to vectorize your single-frame function `gate_single_frame(frame, threshold)` across all `frames`!

In [7]:
@jax.jit
def pre_emphasis_scan(waveform: jax.Array, alpha: float = 0.97) -> jax.Array:
    def fope_filter(carry, x_t):
        y_t = x_t - alpha * carry
        return (x_t, y_t)  # (new_carry, output_at_t)

    init_env = jnp.array(0.0, dtype=waveform.dtype)
    final_carry, y = jax.lax.scan(fope_filter, init_env, waveform)
    return y


def gate_single_frame(frame: jax.Array, threshold: float) -> jax.Array:
    rms_energy = jnp.sqrt(jnp.mean(frame ** 2))
    return jax.lax.cond(
        rms_energy >= threshold,
        lambda x: x,                  # true_fun: return frame unchanged
        lambda x: jnp.zeros_like(x),  # false_fun: return all zeros
        frame                         # operand passed as `x` to both functions
    )

# Vectorize over axis 0 of `frames` (num_frames, frame_len), keeping `threshold` unbatched (None)
batched_noise_gate = jax.jit(jax.vmap(gate_single_frame, in_axes=(0, None)))